In [ ]:
import os
import csv
import ast
import re
import time
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV, KFold
from sklearn.metrics import accuracy_score, hamming_loss, precision_recall_fscore_support
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.multiclass import OneVsRestClassifier

In [10]:
# Variables globales:
SEED = 2627
DEV_PART = 0.15
CORPUS_DIR = "data"
TRAIN_FILE = os.path.join(CORPUS_DIR, "dataset_train.csv")
TEST_FILE = os.path.join(CORPUS_DIR, "dataset_test.csv")
TOKEN_PATTERN = r"\S+" # Captura cualquier grupo de caracteres que NO contenga espacios

In [11]:
# Función para cargar el corpus:
def load_corpus(filename):
    papers, terms = [], []

    # Abrimos el archivo 'filename' y lo leemos línea por línea:
    with open(filename, mode='r', encoding='utf-8') as file:
        reader = csv.reader(file, delimiter=',')

        # Saltamos la fila con los títulos de las columnas:
        next(reader)

        # Extraemos las columnas del archivo:
        for row in reader:
            if not row or len(row) < 3:
                continue

            papers.append((row[0], row[1]))
            terms.append(ast.literal_eval(row[2]))

    return papers, terms


# Función de tokenización:
def tokenize(text):
    # Buscamos palabras que empiecen por letras y puedan contener guiones
    # (para evitar ruido como "5k" y no excluir estructuras como "state-of-the-art"):
    pattern = re.compile(r"(?u)\b[a-zA-Z][\w-]+\b")
    return " ".join(pattern.findall(text))


# Función para calcular las métricas de evaluación:
def compute_metrics(labels, preds):
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, preds, average="macro", zero_division=0
    )
    acc = accuracy_score(labels, preds)
    hl = hamming_loss(labels, preds)

    return {
        "accuracy": acc,
        "f1": f1,
        "precision": precision,
        "recall": recall,
        "hamming_loss": hl,
    }

In [12]:
# Cargamos el corpus, y guardamos las entradas y los términos:
raw_all, y_all = load_corpus(TRAIN_FILE)
print("Papers de entrenamiento:", len(raw_all))

# Convertimos los términos en listas binarias:
mlb = MultiLabelBinarizer()
y_bin = mlb.fit_transform(y_all)

# Creamos una partición aleatoria de entrenamiento/desarrollo:
raw_train, raw_dev, y_train, y_dev = train_test_split(raw_all, y_bin, test_size=DEV_PART, shuffle=True, random_state=SEED)

# Tokenizamos los títulos y los resúmenes:
train_tok = [tokenize(title) + tokenize(summary) for title, summary in raw_train]
dev_tok = [tokenize(title) + tokenize(summary) for title, summary in raw_dev]

Papers de entrenamiento: 43555


In [16]:
import collections

def show_stats(texts, labels, name):
    print(f"{name}\n" + "-" * len(name))
    ntok = [len(t.split()) for t in texts]
    print(f"Muestras: {len(texts)}   Tokens: {sum(ntok)}   Tokens por muestra: {np.mean(ntok):.2f}")
    ct = collections.Counter(labels)
    print("  ".join(f"{k}: {ct[k]} ({100 * ct[k] / len(labels):.1f} %)" for k in sorted(ct)))
    print()

In [19]:
labels_train = [cat for paper_cats in mlb.inverse_transform(y_train) for cat in paper_cats]
labels_dev = [cat for paper_cats in mlb.inverse_transform(y_dev) for cat in paper_cats]

show_stats(train_tok, labels_train, "Entrenamiento")
show_stats(dev_tok, labels_dev, "Validación")

Entrenamiento
-------------
Muestras: 37021   Tokens: 6484882   Tokens por muestra: 175.17
cs.AI: 5137 (8.0 %)  cs.CL: 901 (1.4 %)  cs.CR: 283 (0.4 %)  cs.CV: 23005 (35.7 %)  cs.CY: 82 (0.1 %)  cs.DC: 86 (0.1 %)  cs.GR: 288 (0.4 %)  cs.HC: 38 (0.1 %)  cs.IR: 153 (0.2 %)  cs.IT: 44 (0.1 %)  cs.LG: 19222 (29.9 %)  cs.MA: 186 (0.3 %)  cs.MM: 237 (0.4 %)  cs.NE: 525 (0.8 %)  cs.RO: 1142 (1.8 %)  cs.SI: 358 (0.6 %)  cs.SY: 74 (0.1 %)  eess.IV: 1527 (2.4 %)  eess.SP: 253 (0.4 %)  eess.SY: 74 (0.1 %)  math.IT: 44 (0.1 %)  math.OC: 246 (0.4 %)  math.ST: 37 (0.1 %)  physics.chem-ph: 31 (0.0 %)  stat.AP: 52 (0.1 %)  stat.ME: 44 (0.1 %)  stat.ML: 10281 (16.0 %)  stat.TH: 37 (0.1 %)

Validación
----------
Muestras: 6534   Tokens: 1145100   Tokens por muestra: 175.25
cs.AI: 899 (7.9 %)  cs.CL: 151 (1.3 %)  cs.CR: 45 (0.4 %)  cs.CV: 4073 (35.8 %)  cs.CY: 18 (0.2 %)  cs.DC: 16 (0.1 %)  cs.GR: 50 (0.4 %)  cs.HC: 10 (0.1 %)  cs.IR: 16 (0.1 %)  cs.IT: 5 (0.0 %)  cs.LG: 3379 (29.7 %)  cs.MA: 29 (0.3 %)  

In [38]:
print(compute_metrics(y_dev, np.zeros_like(y_dev)))
print(compute_metrics(y_dev, np.ones_like(y_dev)))

{'accuracy': 0.0, 'f1': 0.0, 'precision': 0.0, 'recall': 0.0, 'hamming_loss': 0.06210918710918711}
{'accuracy': 0.0, 'f1': 0.08888139346828376, 'precision': 0.062109187109187114, 'recall': 1.0, 'hamming_loss': 0.9378908128908129}


In [ ]:
vec = TfidfVectorizer(token_pattern=TOKEN_PATTERN, sublinear_tf=True)
A = vec.fit_transform(train_tok)
B = vec.transform(dev_tok)

multilabel = {}
for c in [0.1, 0.25, 0.5, 0.75, 1, 3, 5, 6, 7, 8, 9, 10]:
    multilabel[c] = OneVsRestClassifier(LogisticRegression(solver="liblinear", C=c, class_weight="balanced", random_state=SEED)).fit(A, y_train)
    y_pred = multilabel[c].predict(B)
    print(f"=== C={c}")
    print(compute_metrics(y_dev, y_pred))

=== C=0.1, 0.1
{'accuracy': 0.25068870523415976, 'f1': 0.3828775210093977, 'precision': 0.31286728597454694, 'recall': 0.6942435956382225, 'hamming_loss': 0.046564126109580654}
=== C=0.25, 0.25
{'accuracy': 0.3111417202326293, 'f1': 0.39915850688501014, 'precision': 0.33524251235804814, 'recall': 0.6494154681800712, 'hamming_loss': 0.04026192662556299}
=== C=0.5, 0.5
{'accuracy': 0.37327823691460055, 'f1': 0.43541858424590946, 'precision': 0.38825915061601834, 'recall': 0.6411783595984153, 'hamming_loss': 0.03542459224277406}
=== C=0.75, 0.75
{'accuracy': 0.4097030915212733, 'f1': 0.4547737544094469, 'precision': 0.41272095939201936, 'recall': 0.6301159019055236, 'hamming_loss': 0.0327517600244873}
=== C=1, 1
{'accuracy': 0.43357820630547905, 'f1': 0.4639510064438256, 'precision': 0.4297905105335521, 'recall': 0.6102667049626519, 'hamming_loss': 0.030969871878962787}
=== C=3, 3
{'accuracy': 0.5238751147842057, 'f1': 0.5138451094128209, 'precision': 0.5954641341570015, 'recall': 0.57326

In [ ]:
vec = TfidfVectorizer(token_pattern=TOKEN_PATTERN, sublinear_tf=True)
A = vec.fit_transform(train_tok)
B = vec.transform(dev_tok)

for c100 in range(100, 300, 25):
    c = c100 / 100
    multilabel[c] = OneVsRestClassifier(LogisticRegression(solver="liblinear", C=c, class_weight="balanced", random_state=SEED)).fit(A, y_train)
    y_pred = multilabel[c].predict(B)
    print(f"=== C={c}, {c}")
    print(compute_metrics(y_dev, y_pred))

=== C=1.0, 1.0
{'accuracy': 0.43357820630547905, 'f1': 0.4639510064438256, 'precision': 0.4297905105335521, 'recall': 0.6102667049626519, 'hamming_loss': 0.030969871878962787}
=== C=1.25, 1.25
{'accuracy': 0.45454545454545453, 'f1': 0.4741469755451201, 'precision': 0.4488947523502235, 'recall': 0.6001223513153785, 'hamming_loss': 0.02954873409418864}
=== C=1.5, 1.5
{'accuracy': 0.465564738292011, 'f1': 0.47866905726333486, 'precision': 0.4564828616775423, 'recall': 0.5959096612613257, 'hamming_loss': 0.028657790021426384}
=== C=1.75, 1.75
{'accuracy': 0.47796143250688705, 'f1': 0.489565304352822, 'precision': 0.47838506882445003, 'recall': 0.5931420461969061, 'hamming_loss': 0.02784336875245966}
=== C=2.0, 2.0
{'accuracy': 0.48806244260789716, 'f1': 0.49213598038279266, 'precision': 0.48277473410530447, 'recall': 0.5889316842891587, 'hamming_loss': 0.02717106126197035}
=== C=2.25, 2.25
{'accuracy': 0.4989286807468626, 'f1': 0.497357754397144, 'precision': 0.49667765169345357, 'recall':

In [42]:
vec = TfidfVectorizer(token_pattern=TOKEN_PATTERN, sublinear_tf=True)
A = vec.fit_transform(train_tok)
B = vec.transform(dev_tok)

for c100 in range(250, 275, 5):
    c = c100 / 100
    multilabel[c] = OneVsRestClassifier(LogisticRegression(solver="liblinear", C=c, class_weight="balanced", random_state=SEED)).fit(A, y_train)
    y_pred = multilabel[c].predict(B)
    print(f"=== C={c}, {c}")
    print(compute_metrics(y_dev, y_pred))

=== C=2.5, 2.5
{'accuracy': 0.5071931435567799, 'f1': 0.500750922198366, 'precision': 0.5140615258117424, 'recall': 0.5790022862295928, 'hamming_loss': 0.026077878350605623}
=== C=2.55, 2.55
{'accuracy': 0.509641873278237, 'f1': 0.5037874182855876, 'precision': 0.5266271249278097, 'recall': 0.5786663224542475, 'hamming_loss': 0.025946696401241857}
=== C=2.6, 2.6
{'accuracy': 0.5113253749617386, 'f1': 0.5061169881979766, 'precision': 0.5514700010458129, 'recall': 0.5787049985127677, 'hamming_loss': 0.025842844024662207}
=== C=2.65, 2.65
{'accuracy': 0.512243648607285, 'f1': 0.5104979213073516, 'precision': 0.588563349393221, 'recall': 0.5784142118365505, 'hamming_loss': 0.02575538939175303}
=== C=2.7, 2.7
{'accuracy': 0.5139271502907866, 'f1': 0.5110570925513228, 'precision': 0.589402790042684, 'recall': 0.5780076089870085, 'hamming_loss': 0.025646071100616556}


In [13]:
from sklearn.svm import LinearSVC

vectorizer = TfidfVectorizer(
    token_pattern=TOKEN_PATTERN,
    ngram_range=(1, 2),
    sublinear_tf=True,
    min_df=3,
    max_features=25000,
)
X_train = vectorizer.fit_transform(train_tok)
X_dev = vectorizer.transform(dev_tok)

for c in [0.05, 0.1, 0.2, 0.3, 0.5, 0.7, 1.0]:
    svm = OneVsRestClassifier(
        LinearSVC(
            C=c,
            dual=False,
            class_weight="balanced",
            max_iter=8000,
            tol=1e-3,
            random_state=SEED,
        ),
        n_jobs=-1,
    )
    svm.fit(X_train, y_train)

    y_pred = svm.predict(X_dev)
    metricas = compute_metrics(y_dev, y_pred)
    print(f"** C = {c}: **\n{metricas}")

** C = 0.05: **
{'accuracy': 0.40404040404040403, 'f1': 0.47312112402855305, 'precision': 0.554330646889048, 'recall': 0.5743006302336433, 'hamming_loss': 0.032735362280816825}
** C = 0.1: **
{'accuracy': 0.46250382614018976, 'f1': 0.5058987670087542, 'precision': 0.618609979135336, 'recall': 0.5510877521815656, 'hamming_loss': 0.02836809654991473}
** C = 0.2: **
{'accuracy': 0.5212733394551576, 'f1': 0.5349563173777655, 'precision': 0.6761771886066005, 'recall': 0.5334211456373648, 'hamming_loss': 0.02491910446455901}
** C = 0.3: **
{'accuracy': 0.5501989592898684, 'f1': 0.5445386083168936, 'precision': 0.7125392960204274, 'recall': 0.5163932885524561, 'hamming_loss': 0.02324106869561415}
** C = 0.5: **
{'accuracy': 0.581267217630854, 'f1': 0.5608064410877418, 'precision': 0.7710954956748013, 'recall': 0.5075629433886081, 'hamming_loss': 0.021727010363374}
** C = 0.7: **
{'accuracy': 0.6020814202632384, 'f1': 0.5618556733048241, 'precision': 0.7916977039900904, 'recall': 0.49732322031

In [14]:
for c in [0.6, 0.8, 0.9]:
    svm = OneVsRestClassifier(
        LinearSVC(
            C=c,
            dual=False,
            class_weight="balanced",
            max_iter=8000,
            tol=1e-3,
            random_state=SEED,
        ),
        n_jobs=-1,
    )
    svm.fit(X_train, y_train)

    y_pred = svm.predict(X_dev)
    metricas = compute_metrics(y_dev, y_pred)
    print(f"** C = {c}: **\n{metricas}")

** C = 0.6: **
{'accuracy': 0.5930517294153658, 'f1': 0.5652170919738506, 'precision': 0.7878031171363766, 'recall': 0.5044286884461882, 'hamming_loss': 0.02115855524946434}
** C = 0.8: **
{'accuracy': 0.608356290174472, 'f1': 0.563008414255246, 'precision': 0.7971403475019396, 'recall': 0.49498532381869326, 'hamming_loss': 0.020447986357077265}
** C = 0.9: **
{'accuracy': 0.6138659320477502, 'f1': 0.5607706079860381, 'precision': 0.8015712512535824, 'recall': 0.48809945307152686, 'hamming_loss': 0.020191088372906556}


In [15]:
for c in [0.6, 0.63, 0.66, 0.68]:
    svm = OneVsRestClassifier(
        LinearSVC(
            C=c,
            dual=False,
            class_weight="balanced",
            max_iter=8000,
            tol=1e-3,
            random_state=SEED,
        ),
        n_jobs=-1,
    )
    svm.fit(X_train, y_train)

    y_pred = svm.predict(X_dev)
    metricas = compute_metrics(y_dev, y_pred)
    print(f"** C = {c}: **\n{metricas}")

** C = 0.6: **
{'accuracy': 0.5930517294153658, 'f1': 0.5652170919738506, 'precision': 0.7878031171363766, 'recall': 0.5044286884461882, 'hamming_loss': 0.02115855524946434}
** C = 0.63: **
{'accuracy': 0.5962656871747781, 'f1': 0.5633874677167723, 'precision': 0.7889198981989681, 'recall': 0.5009058457697028, 'hamming_loss': 0.021016441470986927}
** C = 0.66: **
{'accuracy': 0.5982552800734618, 'f1': 0.5638386574042457, 'precision': 0.7900612542936413, 'recall': 0.5006799209452472, 'hamming_loss': 0.020885259521623158}
** C = 0.68: **
{'accuracy': 0.60085705540251, 'f1': 0.5613047035476034, 'precision': 0.7896924713700919, 'recall': 0.49759322401456874, 'hamming_loss': 0.020803270803270803}
